In [ ]:
import re
import pandas as pd
import os

# 1. Define the input file path
# Note: Use raw string (r"...") to handle backslashes in Windows paths correctly
file_path = r"C:\Users\dhrub\OneDrive\Desktop\Mayo-CoRAL\LLM_Extraction_and_CrossCritique\interactive-table\runs\file2txt.txt"

# 2. Define the output CSV path (saving it in the same directory as input)
output_path = os.path.join(os.path.dirname(file_path), "extracted_questions.csv")

def parse_questions_to_csv(input_file, output_file):
    # Check if file exists
    if not os.path.exists(input_file):
        print(f"Error: File not found at {input_file}")
        return

    # Read the file content
    with open(input_file, 'r', encoding='utf-8') as f:
        content = f.read()

    # 3. Define the Regex Pattern
    # This pattern looks for:
    # - ### Question [number]
    # - **Query:** [text content until the next marker]
    # - **Ground Truth JSON:** ```json [json content] ```
    pattern = re.compile(
        r"### Question (\d+)\s+"             # Capture Group 1: Question Number
        r"\*\*Query:\*\*\s+"                 # Match Query header
        r"(.*?)\s+"                          # Capture Group 2: Query text (non-greedy)
        r"\*\*Ground Truth JSON:\*\*\s+"     # Match JSON header
        r"```json\s+"                        # Match code block start
        r"(.*?)\s+"                          # Capture Group 3: JSON content (non-greedy)
        r"```",                              # Match code block end
        re.DOTALL                            # DOTALL allows . to match newlines
    )

    # Find all matches
    matches = pattern.findall(content)

    if not matches:
        print("No matches found. Please check the file formatting.")
        return

    # 4. Create a DataFrame
    data = []
    for match in matches:
        q_num, query_text, json_text = match
        data.append({
            "Question Number": q_num.strip(),
            "Query": query_text.strip(),
            "Ground Truth JSON": json_text.strip()
        })

    df = pd.DataFrame(data)

    # 5. Save to CSV
    # escaping is handled automatically by pandas (e.g., quotes inside JSON)
    df.to_csv(output_file, index=False, encoding='utf-8')

    print(f"Successfully extracted {len(df)} questions.")
    print(f"Saved CSV to: {output_file}")
    
    # Optional: Preview the first few rows
    print("\nPreview:")
    print(df.head())

# Run the function
parse_questions_to_csv(file_path, output_path)